# Georgia Election Retrieval

Use on election results stored at this portal: https://app.enhancedvoting.com/results/public/Georgia

Open the link for the particular election (ex. https://app.enhancedvoting.com/results/public/Georgia/elections/GeneralPrimary51926)

Rename the ELECTION variable below as the last slash in the url
Rename the output directory file


In [1]:
import os
import requests

ELECTION = "06162026GeneralPrimaryRunoff"
OUT_DIR = "GA_Results_Prim_RunoffJune16_26"

os.makedirs(OUT_DIR, exist_ok=True)

session = requests.Session()
session.headers["User-Agent"] = "Mozilla/5.0"

# ---------------------------------------------------------------------
# Get participating counties from the statewide page
# ---------------------------------------------------------------------

state_url = (
    f"https://app.enhancedvoting.com/results/public/api/"
    f"elections/Georgia/{ELECTION}/data"
)

state_data = session.get(state_url, timeout=20).json()

# Build lookup of participating counties
localities = state_data["jurisdiction"]["childLocalities"]

lookup = {
    loc["shortName"]: loc["id"].lower()
    for loc in localities
}

print(f"Found {len(lookup)} participating counties.\n")

# ---------------------------------------------------------------------
# Download reports
# ---------------------------------------------------------------------

for short_name, folder in lookup.items():

    county = short_name.replace("-county-ga", "")

    print(f"Processing {county}...")

    api_url = (
        f"https://app.enhancedvoting.com/results/public/api/"
        f"elections/{short_name}/{ELECTION}/data"
    )

    try:
        data = session.get(api_url, timeout=20).json()

        # Find the Total Votes Results report
        blob = None
        for category in data["election"]["publicReportCategories"]:
            for report in category["reports"]:
                if report["blobName"].startswith("Total Votes Results"):
                    blob = report["blobName"]
                    break
            if blob:
                break

        if blob is None:
            print("  No Total Votes Results report found.")
            continue

        download_url = (
            f"https://app.enhancedvoting.com/cdn/results/"
            f"{folder}/{blob}"
        )

        r = session.get(download_url, timeout=60)
        r.raise_for_status()

        out_file = os.path.join(OUT_DIR, f"{county}.xlsx")

        with open(out_file, "wb") as f:
            f.write(r.content)

        print(f"  Saved {out_file}")

    except Exception as e:
        print(f"  ERROR: {e}")

Found 159 participating counties.

Processing appling...
  Saved GA_Results_Prim_RunoffJune16_26/appling.xlsx
Processing atkinson...
  Saved GA_Results_Prim_RunoffJune16_26/atkinson.xlsx
Processing bacon...
  Saved GA_Results_Prim_RunoffJune16_26/bacon.xlsx
Processing baker...
  Saved GA_Results_Prim_RunoffJune16_26/baker.xlsx
Processing baldwin...
  Saved GA_Results_Prim_RunoffJune16_26/baldwin.xlsx
Processing banks...
  Saved GA_Results_Prim_RunoffJune16_26/banks.xlsx
Processing barrow...
  Saved GA_Results_Prim_RunoffJune16_26/barrow.xlsx
Processing bartow...
  Saved GA_Results_Prim_RunoffJune16_26/bartow.xlsx
Processing ben-hill...
  Saved GA_Results_Prim_RunoffJune16_26/ben-hill.xlsx
Processing berrien...
  Saved GA_Results_Prim_RunoffJune16_26/berrien.xlsx
Processing bibb...
  Saved GA_Results_Prim_RunoffJune16_26/bibb.xlsx
Processing bleckley...
  Saved GA_Results_Prim_RunoffJune16_26/bleckley.xlsx
Processing brantley...
  Saved GA_Results_Prim_RunoffJune16_26/brantley.xlsx
Proc

  Saved GA_Results_Prim_RunoffJune16_26/peach.xlsx
Processing pickens...
  Saved GA_Results_Prim_RunoffJune16_26/pickens.xlsx
Processing pierce...
  Saved GA_Results_Prim_RunoffJune16_26/pierce.xlsx
Processing pike...
  Saved GA_Results_Prim_RunoffJune16_26/pike.xlsx
Processing polk...
  Saved GA_Results_Prim_RunoffJune16_26/polk.xlsx
Processing pulaski...
  Saved GA_Results_Prim_RunoffJune16_26/pulaski.xlsx
Processing putnam...
  Saved GA_Results_Prim_RunoffJune16_26/putnam.xlsx
Processing quitman...
  Saved GA_Results_Prim_RunoffJune16_26/quitman.xlsx
Processing rabun...
  Saved GA_Results_Prim_RunoffJune16_26/rabun.xlsx
Processing randolph...
  Saved GA_Results_Prim_RunoffJune16_26/randolph.xlsx
Processing richmond...
  Saved GA_Results_Prim_RunoffJune16_26/richmond.xlsx
Processing rockdale...
  Saved GA_Results_Prim_RunoffJune16_26/rockdale.xlsx
Processing schley...
  Saved GA_Results_Prim_RunoffJune16_26/schley.xlsx
Processing screven...
  Saved GA_Results_Prim_RunoffJune16_26/scr

In [9]:
# import os
# import requests

# ELECTION = "06092026SpecialElectionRunoff"
# OUT_DIR = "GA_Results_Prim_Runoff26"

# # List of Georgia counties (replace with your full list)
# counties = [
# "appling","atkinson","bacon","baker","baldwin","banks","barrow","bartow",
#     "ben-hill","berrien","bibb","bleckley","brantley","brooks","bryan","bulloch",
#     "burke","butts","calhoun","camden","candler","carroll","catoosa","charlton",
#     "chatham","chattahoochee","chattooga","cherokee","clarke","clay","clayton",
#     "clinch","cobb","coffee","colquitt","columbia","cook","coweta","crawford",
#     "crisp","dade","dawson","decatur","dekalb","dodge","dooly","dougherty",
#     "douglas","early","echols","effingham","elbert","emanuel","evans","fannin",
#     "fayette","floyd","forsyth","franklin","fulton","gilmer","glascock","glynn",
#     "gordon","grady","greene","gwinnett","habersham","hall","hancock","haralson",
#     "harris","hart","heard","henry","houston","irwin","jackson","jasper","jeff-davis",
#     "jefferson","jenkins","johnson","jones","lamar","lanier","laurens","lee",
#     "liberty","lincoln","long","lowndes","lumpkin","macon","madison","marion",
#     "mcduffie","mcintosh","meriwether","miller","mitchell","monroe","montgomery",
#     "morgan","murray","muscogee","newton","oconee","oglethorpe","paulding","peach",
#     "pickens","pierce","pike","polk","pulaski","putnam","quitman","rabun",
#     "randolph","richmond","rockdale","schley","screven","seminole","spalding",
#     "stephens","stewart","sumter","talbot","taliaferro","tattnall","taylor",
#     "telfair","terrell","thomas","tift","toombs","towns","treutlen","troup",
#     "turner","twiggs","union","upson","walker","walton","ware","warren",
#     "washington","wayne","webster","wheeler","white","whitfield","wilcox",
#     "wilkes","wilkinson","worth"]




# os.makedirs(OUT_DIR, exist_ok=True)

# session = requests.Session()
# session.headers["User-Agent"] = "Mozilla/5.0"

# # ---------------------------------------------------------------------
# # Get the lookup table that maps county shortName -> UUID folder
# # ---------------------------------------------------------------------

# seed_url = (
#     f"https://app.enhancedvoting.com/results/public/api/"
#     f"elections/appling-county-ga/{ELECTION}/data"
# )

# seed = session.get(seed_url).json()

# lookup = {
#     item["shortName"]: item["id"].lower()
#     for item in seed["jurisdiction"]["parent"]["childLocalities"]
# }

# # ---------------------------------------------------------------------
# # Download reports
# # ---------------------------------------------------------------------

# for county in counties:

#     print(f"Processing {county}...")

#     api_url = (
#         f"https://app.enhancedvoting.com/results/public/api/"
#         f"elections/{county}-county-ga/{ELECTION}/data"
#     )

#     try:
#         data = session.get(api_url, timeout=20).json()

#         reports = data["election"]["publicReportCategories"][0]["reports"]

#         if len(reports) == 0:
#             print("  No reports found.")
#             continue

#         blob = reports[0]["blobName"]

#         folder = lookup[f"{county}-county-ga"]

#         download_url = (
#             f"https://app.enhancedvoting.com/cdn/results/"
#             f"{folder}/{blob}"
#         )

#         r = session.get(download_url, timeout=60)
#         r.raise_for_status()

#         out_file = os.path.join(OUT_DIR, f"{county}.xlsx")

#         with open(out_file, "wb") as f:
#             f.write(r.content)
#         print(f"  Saved {out_file}")

#     except Exception as e:
#         print(f"  ERROR: {e}")

JSONDecodeError: Expecting value: line 1 column 1 (char 0)